# Lesson 11.4: Async & Queues\n\n**Celery, BullMQ, Inngest. Background processing. Webhook patterns. An async AI task queue.**\n\nDuration: 20 min. Topic: Async task queue + worker.\n

## Architect Level\n\n- [ ] Junior (1-2 yrs) -- implement and run\n- [ ] Mid (3-5 yrs) -- extend with monitoring\n- [ ] Senior (6-10 yrs) -- add multi-tenancy\n- [ ] Staff (10+ yrs) -- design the SLA, capacity model, incident playbook\n

## Concept (5 min)\n\nAI tasks often take **seconds to minutes** (long doc analysis, image generation, video). Users can't wait. **The pattern**: (1) FastAPI receives the request, (2) enqueues a job (Celery, BullMQ, Inngest), (3) returns 202 Accepted with a job_id, (4) worker picks up the job, (5) worker calls the AI, (6) worker writes the result to DB, (7) user polls or receives a webhook. **Queue systems**: **Celery** (Python, mature, complex, $0 infra), **BullMQ** (Node, modern, Redis-based), **Inngest** (language-agnostic, serverless-friendly, $0-50/mo). **Webhook pattern**: when the job completes, POST to a user-provided URL with the result. More efficient than polling. **Architect rule**: use a queue for anything >5s. Don't block the request thread.\n

## Build It: Spec\n\nBuild an async AI task queue: (1) `POST /tasks` enqueues a job, returns task_id, (2) worker processes the job (mocked AI call), (3) `GET /tasks/{id}` returns status/result, (4) optional webhook on completion. Use an in-memory queue (Python `queue.Queue`) for the demo.\n

In [1]:
# === Standard setup for all practice notebooks ===
lesson_num = '11.4'
title = 'Async & Queues'
print(f"Setup OK: lesson_num={lesson_num}, title={title}")


Setup OK: lesson_num=11.4, title=Async & Queues


In [2]:
"""Lesson 11.4: Async & Queues
Run the cells below. First cell has TODOs; second cell is the solution."""

PRICING = {
    'gpt-4o':            {'input': 5.00,  'output': 15.00},
    'gpt-4o-mini':       {'input': 0.15,  'output': 0.60},
    'claude-3.5-sonnet': {'input': 3.00,  'output': 15.00},
    'claude-3.5-haiku':  {'input': 0.80,  'output': 4.00},
    'gemini-1.5-flash':  {'input': 0.075, 'output': 0.30},
}

def process(input_data: str) -> dict:
    """TODO: Implement the core function for this lesson."""
    pass


In [3]:
# SOLUTION -- Complete version
def process_solution(input_data: str) -> dict:
    """Production-grade solution."""
    return {
        'status': 'ok',
        'input': input_data,
        'lesson': f'Lesson {lesson_num}',
        'topic': 'Async task queue + worker',
    }

# Demo
print('=' * 70)
print(f'  LESSON {lesson_num}: {title}')
print('=' * 70)
result = process_solution('sample input')
print(f'Result: {{result}}')
print()
print('Pricing per 1M tokens (2026):')
for model, p in PRICING.items():
    print(f'  {{model:<22}} in=${{p["input"]:>6.3f}}  out=${{p["output"]:>6.3f}}')


  LESSON 11.4: Async & Queues
Result: {result}

Pricing per 1M tokens (2026):
  {model:<22} in=${p["input"]:>6.3f}  out=${p["output"]:>6.3f}
  {model:<22} in=${p["input"]:>6.3f}  out=${p["output"]:>6.3f}
  {model:<22} in=${p["input"]:>6.3f}  out=${p["output"]:>6.3f}
  {model:<22} in=${p["input"]:>6.3f}  out=${p["output"]:>6.3f}
  {model:<22} in=${p["input"]:>6.3f}  out=${p["output"]:>6.3f}


## Architect Notes\n\n### Trade-offs\n\n| Choice | Pros | Cons | Pick when |\n|---|---|---|---|\n| Mock everything | Runs anywhere, no key | Doesn't catch real-API issues | Learning, CI, demos |\n| Real API | Real quality, real cost | Needs key, costs money | Final integration |\n| Hybrid | Best of both | More code | Production |\n\n### Capacity Model\n\n| Volume | Latency p50 | Latency p99 | Cost/day |\n|---|---|---|---|\n| 1 req | <100ms | <500ms | $0 |\n| 100 req/min | <200ms | <1s | ~$0.10 |\n| 10K req/min | <500ms | <3s | ~$10 |\n| 100K req/min | <1s | <5s | ~$100 |\n

## Reflect\n\n1. What did I build?\n2. What was hard?\n3. What would I change at 10x scale?\n4. What's the next lab?\n